# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subata24/ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Question

*The research question and the decision it supports.*

###Which content pages are most likely to be declining in organic search performance, and should therefore be prioritized for refresh — supporting a content editor's decision of where to spend limited update effort first.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank internship warehouse (Hugging Face, `FlyRank/internship-warehouse`, gated access).

**Tables:** `fact_content_daily_performance` (daily GSC/GA4 metrics, one row per client × content × day) and `dim_content` (static page-level metadata: word count, search volume, content creation date).

**Date windows:** Development and evaluation on `month=2026-02` and `month=2026-03` (mid-panel months) — February used as the "before" snapshot for features, March as the "after" snapshot the label is measured against. The dataset's final month (`month=2026-06` / the `_sample` table) was never used, since it is the natural outcome window for any past→future label and using it during development would mean testing inside the same window used to build the label logic.

**Grain:** One row = one content page (a client + content identifier pair), summarized at the month level from daily source rows.

**Exclusions (with why):**
- Page-months with no GSC data available (`gsc_data_available` false) — no way to measure clicks, so no way to measure decline.
- Pages with fewer than 5 clicks in February — near-zero traffic makes "decline" mechanically undefined (a page with 0 clicks cannot go lower), so measuring decline on it produces a floor artifact rather than a real signal.
- Editorial/system metadata (e.g. which AI tool generated a page, whether a page has already been marked eligible for optimization) — these risk encoding a human decision about this exact question, rather than describing the page's organic performance.

All identifiers used throughout (client and content references) are salted, namespaced hash keys — used only for grouping and joining, never as model features, and no raw client names, domains, or query text appear anywhere in this work.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [2]:
%pip install -q duckdb huggingface_hub

import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{}');".format(os.environ["HF_TOKEN"]))

MIN_CLICKS_FEB = 5

label_df = con.sql(f"""
WITH feb AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_feb
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
mar AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_mar
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    mar.client_hash_id,
    mar.content_hash_id,
    feb.clicks_feb,
    mar.clicks_mar,
    CASE WHEN mar.clicks_mar < feb.clicks_feb THEN 1 ELSE 0 END AS declined
FROM mar
JOIN feb ON mar.client_hash_id = feb.client_hash_id AND mar.content_hash_id = feb.content_hash_id
WHERE feb.clicks_feb >= {MIN_CLICKS_FEB}
""").df()

print("Rows after clicks_feb >= 5 filter:", label_df.shape)
print("Decline rate:", label_df["declined"].mean())
print("Min clicks_feb in data:", label_df["clicks_feb"].min())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after clicks_feb >= 5 filter: (21776, 5)
Decline rate: 0.4395205731080088
Min clicks_feb in data: 5.0


In [3]:
con.register("label_df_tbl", label_df)

features_df = con.sql("""
WITH feb_metrics AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position_feb,
        SUM(sessions_organic) AS sessions_organic_feb,
        SUM(ga4_engaged_sessions) AS engaged_sessions_feb,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_impressions) AS feb_impressions
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    fm.client_hash_id,
    fm.content_hash_id,
    fm.avg_position_feb,
    fm.sessions_organic_feb,
    fm.engaged_sessions_feb,
    dc.word_count,
    dc.search_volume,
    DATE_DIFF('day', dc.content_created_date, DATE '2026-03-01') AS content_age_days,
    CASE WHEN fm.feb_impressions > 0 THEN fm.feb_clicks * 1.0 / fm.feb_impressions ELSE NULL END AS ctr,
    CASE
        WHEN fm.avg_position_feb <= 3 THEN 0.25
        WHEN fm.avg_position_feb <= 10 THEN 0.05
        ELSE 0.01
    END AS expected_ctr
FROM feb_metrics fm
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' dc
    ON fm.client_hash_id = dc.client_hash_id AND fm.content_hash_id = dc.content_hash_id
JOIN label_df_tbl l
    ON fm.client_hash_id = l.client_hash_id AND fm.content_hash_id = l.content_hash_id
WHERE dc.content_created_date IS NOT NULL
""").df()

features_df["ctr_gap"] = features_df["ctr"] - features_df["expected_ctr"]

print(features_df.shape)
features_df.isna().sum()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(21776, 11)


,0
client_hash_id,0
content_hash_id,0
avg_position_feb,0
sessions_organic_feb,10561
engaged_sessions_feb,10561
word_count,3569
search_volume,289
content_age_days,0
ctr,0
expected_ctr,0


In [4]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

merged = label_df.merge(features_df, on=["client_hash_id", "content_hash_id"])
merged = merged.dropna(subset=["avg_position_feb", "word_count", "search_volume"])

feature_cols = ["avg_position_feb", "sessions_organic_feb", "engaged_sessions_feb",
                "word_count", "search_volume", "content_age_days", "ctr_gap"]

X = merged[feature_cols].fillna(0)
y = merged["declined"]
groups = merged["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Overlap check (should be 0):", len(set(merged.iloc[train_idx]['client_hash_id']) & set(merged.iloc[test_idx]['client_hash_id'])))
print("Base rate (test):", y_test.mean())

Train: (7210, 7) Test: (10747, 7)
Overlap check (should be 0): 0
Base rate (test): 0.41825625756024937


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
